# **CatBoost**

это библиотека градиентного бустинга по деревьям решений от Яндекса, специально заточенная под:

табличные данные с категориальными фичами (типы сигналов, коды, id-шки, города и т.п.),

минимальный ручной feature engineering:
«скормил сырую таблицу → уже работает»,

уменьшение переобучения из-за утечек.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [ ]:
!pip install catboost

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.6 MB/s eta 0:00:00


Таргет Good trade (То есть ret_H — вспомогательная переменная для вычисления GoodTrade)


In [ ]:
# путь подставь свой, если файл называется иначе
df = pd.read_csv("Brent.csv")

# Горизонт удержания сделки (можешь менять, например 10, 20, 30 баров)
H = 20

# Будущая цена
df["Close_fwd"] = df["Close"].shift(-H)

# Доходность по направлению сигнала
ret_long = (df["Close_fwd"] - df["Close"]) / df["Close"]
ret_short = (df["Close"] - df["Close_fwd"]) / df["Close"]

ret_H = np.where(
    df["EntrySignal"] > 0, ret_long,
    np.where(df["EntrySignal"] < 0, ret_short, 0.0)
)

df["ret_H"] = ret_H

# Таргет: хорошая сделка, если есть сигнал и ret_H > 0
df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

# Убираем последние H строк, где Close_fwd и ret_H некорректны
df = df.iloc[:-H].reset_index(drop=True)

print(df[["Close", "Close_fwd", "ret_H", "GoodTrade"]].head())


   Close  Close_fwd     ret_H  GoodTrade
0  48.09      46.70  0.000000          0
1  48.30      46.77  0.000000          0
2  48.30      46.81  0.030849          1
3  48.09      46.58  0.000000          0
4  48.07      46.77  0.000000          0


Подготовка данных для CatBoost

In [ ]:
# Берём только строки, где есть сигнал (мы фильтруем входы)
df_sig = df[df["EntrySignal"] != 0].copy()

# Заполняем пропуски в EntryReason (если есть)
if "EntryReason" in df_sig.columns:
    df_sig["EntryReason"] = df_sig["EntryReason"].fillna("None").astype(str)

target_col = "GoodTrade"

# Колонки, которые НЕ должны быть фичами
drop_cols = ["DateTime", "Close_fwd", "ret_H"]

feature_cols = [
    c for c in df_sig.columns
    if c not in drop_cols + [target_col]
]

print("Фичи:", feature_cols)

# Категориальные колонки (CatBoost сам с ними справится)
cat_cols = df_sig[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
print("Категориальные фичи:", cat_cols)

# Делим по времени (без shuffle!)
split_idx = int(len(df_sig) * 0.8)

train = df_sig.iloc[:split_idx].copy()
test  = df_sig.iloc[split_idx:].copy()

X_train = train[feature_cols]
y_train = train[target_col]

X_test = test[feature_cols]
y_test = test[target_col]

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Доля GoodTrade=1 в train:", y_train.mean())
print("Доля GoodTrade=1 в test :", y_test.mean())


Фичи: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'EntryReason', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Категориальные фичи: ['EntryReason']
Train size: (1431, 31) Test size: (358, 31)
Доля GoodTrade=1 в train: 0.49056603773584906
Доля GoodTrade=1 в test : 0.48324022346368717


Обучение CatBoost

In [ ]:
model_cb = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=5,
    learning_rate=0.05,
    iterations=500,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=50,          # лог каждые 50 итераций
    use_best_model=True  # запомнит лучшую итерацию по AUC на валидации
)

model_cb.fit(
    X_train, y_train,
    cat_features=cat_cols,        # имена категориальных столбцов
    eval_set=(X_test, y_test)
)


0:	test: 0.5281519	best: 0.5281519 (0)	total: 54.6ms	remaining: 27.2s
50:	test: 0.5276988	best: 0.5301515 (33)	total: 293ms	remaining: 2.58s
100:	test: 0.5414466	best: 0.5419153 (99)	total: 525ms	remaining: 2.08s
150:	test: 0.5463209	best: 0.5468833 (142)	total: 777ms	remaining: 1.79s
200:	test: 0.5517263	best: 0.5552570 (180)	total: 1.01s	remaining: 1.5s
250:	test: 0.5419466	best: 0.5552570 (180)	total: 1.24s	remaining: 1.23s
300:	test: 0.5497266	best: 0.5552570 (180)	total: 1.47s	remaining: 973ms
350:	test: 0.5445712	best: 0.5552570 (180)	total: 1.7s	remaining: 720ms
400:	test: 0.5446649	best: 0.5552570 (180)	total: 1.96s	remaining: 483ms
450:	test: 0.5463209	best: 0.5552570 (180)	total: 2.2s	remaining: 239ms
499:	test: 0.5484768	best: 0.5552570 (180)	total: 2.43s	remaining: 0us

bestTest = 0.5552569911
bestIteration = 180

Shrink model to first 181 iterations.


In [ ]:
proba_cb = model_cb.predict_proba(X_test)[:, 1]
y_pred_cb = (proba_cb >= 0.5).astype(int)

print("CatBoost AUC:", roc_auc_score(y_test, proba_cb))
print("\nОтчёт по классификации (CatBoost):")
print(classification_report(y_test, y_pred_cb))
print("Матрица ошибок (CatBoost):")
print(confusion_matrix(y_test, y_pred_cb))


CatBoost AUC: 0.5552569910951414

Отчёт по классификации (CatBoost):
              precision    recall  f1-score   support

           0       0.54      0.66      0.60       185
           1       0.53      0.40      0.46       173

    accuracy                           0.54       358
   macro avg       0.54      0.53      0.53       358
weighted avg       0.54      0.54      0.53       358

Матрица ошибок (CatBoost):
[[123  62]
 [103  70]]
